# Chapter 8 — RAG Generation
## Topic 3: Faithfulness vs. Relevance vs. Correctness — Three Distinct Failure Modes

**Run cells in order.**

- Module 1: Setup — four scenario answers, each demonstrating a different combination
- Module 2: Faithfulness check (answer vs. retrieved context)
- Module 3: Relevance check (answer vs. original query, via embeddings)
- Module 4: Full decomposition — classify each scenario across all three axes

**Install:** `pip install sentence-transformers numpy` (Module 3 only)


## Module 1: Setup — Four Scenario Answers

Each scenario is DESIGNED to isolate one specific combination of the three failure modes.

**Requires:** nothing

In [ ]:
"""
MODULE 1: Setup

Four scenarios, each engineered to demonstrate a specific combination of
faithfulness / relevance / correctness -- so the checks in later modules
can be validated against a KNOWN expected classification.
"""

RETRIEVED_CONTEXT = {
    "source": "04_FD_Policy_Reference.pdf",
    "text": "Premature withdrawal of Fixed Deposit incurs a 1 percent penalty on the applicable interest rate."
}

QUERY = "What is the penalty for premature FD withdrawal?"

SCENARIOS = [
    {
        "label": "Faithful + Relevant + (assume Correct)",
        "answer": "The penalty for premature FD withdrawal is 1 percent on the applicable interest rate.",
        "expected_faithful": True,
        "expected_relevant": True,
        "note": "Matches context exactly, directly answers the question.",
    },
    {
        "label": "UNFAITHFUL (contradicts the retrieved context)",
        "answer": "The penalty for premature FD withdrawal is 2 percent on the applicable interest rate.",
        "expected_faithful": False,
        "expected_relevant": True,
        "note": "Addresses the right question, but the NUMBER contradicts what was retrieved.",
    },
    {
        "label": "Faithful but IRRELEVANT (retrieval mismatch)",
        "answer": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
        "expected_faithful": None,  # faithful to a DIFFERENT context than what was asked about
        "expected_relevant": False,
        "note": "Accurately reflects SOME content, but does not answer the penalty question asked.",
    },
    {
        "label": "Faithful + Relevant, but context itself may be WRONG (correctness, unverifiable here)",
        "answer": "The penalty for premature FD withdrawal is 1 percent on the applicable interest rate.",
        "expected_faithful": True,
        "expected_relevant": True,
        "note": "Same as scenario 1 -- but imagine the underlying policy doc is 6 months STALE. "
                "Faithfulness and relevance checks CANNOT detect this -- correctness requires "
                "external ground truth, which this pipeline alone cannot provide.",
    },
]

print(f"Query: '{QUERY}'")
print(f"Retrieved context: '{RETRIEVED_CONTEXT['text']}'\n")
for i, s in enumerate(SCENARIOS, start=1):
    print(f"Scenario {i}: {s['label']}")
    print(f"  Answer: '{s['answer']}'")
    print(f"  Note: {s['note']}\n")

print("Module 1 complete. Run Module 2.")


## Module 2: Faithfulness Check

**Requires:** Module 1

In [ ]:
"""
MODULE 2: Faithfulness Check

Simple, illustrative faithfulness check: extract the numeric claim from
the answer and compare against the numeric claim in the retrieved context.
Real production faithfulness checking (Topic 4) uses embedding/NLI/LLM-judge
methods -- this module demonstrates the CONCEPT with a deliberately simple,
fully-offline, deterministic check on this specific numeric-claim pattern.
"""

import re

def extract_percentage(text: str) -> float:
    match = re.search(r"(\d+(?:\.\d+)?)\s*percent", text.lower())
    return float(match.group(1)) if match else None


def check_faithfulness_simple(answer: str, context: str) -> dict:
    """Illustrative check: do the numeric claims match?
    A real system would use embedding/NLI/LLM-judge (Topic 4) for the
    general case -- this demonstrates why even a SIMPLE, deterministic
    check can catch a real unfaithfulness case."""
    answer_pct = extract_percentage(answer)
    context_pct = extract_percentage(context)

    if answer_pct is None or context_pct is None:
        return {"faithful": None, "reason": "No comparable numeric claim found"}

    faithful = answer_pct == context_pct
    return {
        "faithful": faithful,
        "answer_claim": answer_pct,
        "context_claim": context_pct,
        "reason": "Numeric claims match" if faithful else f"MISMATCH: answer says {answer_pct}%, context says {context_pct}%",
    }


print("Faithfulness checks against the SAME retrieved context:\n")
for i, s in enumerate(SCENARIOS, start=1):
    result = check_faithfulness_simple(s["answer"], RETRIEVED_CONTEXT["text"])
    print(f"Scenario {i}: {s['label']}")
    print(f"  {result}")
    print()

print("Module 2 complete. Run Module 3.")


## Module 3: Relevance Check (Query vs. Answer)

**Requires:** Module 1

In [ ]:
"""
MODULE 3: Relevance Check

Independent of what was retrieved -- compares the ANSWER directly against
the ORIGINAL QUERY using embedding similarity. A faithful answer can still
be irrelevant if it doesn't address what was actually asked.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading embedding model (may take a moment on first run)...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


def check_relevance(query: str, answer: str, embed_model) -> float:
    q_vec = embed_model.encode(query, normalize_embeddings=True, show_progress_bar=False)
    a_vec = embed_model.encode(answer, normalize_embeddings=True, show_progress_bar=False)
    return float(np.dot(q_vec, a_vec))


print(f"\nQuery: '{QUERY}'\n")
relevance_scores = []
for i, s in enumerate(SCENARIOS, start=1):
    score = check_relevance(QUERY, s["answer"], embed_model)
    relevance_scores.append(score)
    flag = "LOW (likely irrelevant)" if score < 0.5 else "OK"
    print(f"Scenario {i}: {s['label']}")
    print(f"  Relevance score: {score:.4f} [{flag}]")
    print(f"  Expected relevant: {s['expected_relevant']}")
    print()

print("Module 3 complete. Run Module 4.")


## Module 4: Full Decomposition — Classify Every Scenario

**Requires:** Module 1, Module 2, Module 3

In [ ]:
"""
MODULE 4: Full Decomposition

Combines faithfulness + relevance checks into the full 2x2(x correctness)
classification table from the theory, and explicitly calls out which
combination CANNOT be automatically detected by this pipeline alone.
"""

print(f"{'Scenario':<12} | {'Faithful':<10} | {'Relevant':<10} | {'Correctness (needs ground truth)':<38}")
print("-" * 80)

for i, s in enumerate(SCENARIOS, start=1):
    faith_result = check_faithfulness_simple(s["answer"], RETRIEVED_CONTEXT["text"])
    rel_score = relevance_scores[i - 1]
    is_relevant = rel_score >= 0.5

    faith_str = str(faith_result["faithful"])
    rel_str = str(is_relevant)

    if i == 4:
        correctness_str = "CANNOT AUTOMATE -- requires human/external verification"
    elif i == 3:
        correctness_str = "N/A -- answer addresses wrong topic (relevance failure dominates)"
    else:
        correctness_str = "Assumed correct IF context is current (unverifiable in-pipeline)"

    print(f"Scenario {i:<7} | {faith_str:<10} | {rel_str:<10} | {correctness_str}")

print("\n" + "=" * 70)
print("KEY INSIGHT")
print("=" * 70)
print("""
Scenario 1 and Scenario 4 have IDENTICAL faithfulness and relevance results
(both Faithful=True, Relevant=True) -- yet Scenario 4 represents a case
where the underlying knowledge base could be STALE, making the answer
factually WRONG despite passing every automated check this pipeline can run.

This is the core lesson of this topic: faithfulness and relevance checks
give NO signal about correctness. A system that only monitors these two
metrics can have 100% faithfulness and 100% relevance while silently
serving outdated or wrong information -- correctness requires a
fundamentally different verification mechanism (human review, external
ground truth, knowledge base freshness governance) that lives OUTSIDE
this generation-layer pipeline entirely.
""")

print("=" * 70)
print("CHAPTER 8 TOPIC 3 SUMMARY")
print("=" * 70)
print("""
Faithfulness: does the answer match the RETRIEVED CONTEXT? (Module 2)
Relevance: does the answer match the ORIGINAL QUERY? (Module 3)
Correctness: is the answer actually TRUE? (cannot be automated in-pipeline)

Scenario 2 showed: faithfulness catches numeric-claim contradictions.
Scenario 3 showed: a faithful answer can still be irrelevant (retrieval miss).
Scenario 4 showed: faithful + relevant answers can STILL be wrong if the
  knowledge base itself is stale -- correctness is a distinct, unautomatable
  (within this pipeline) failure mode requiring knowledge base governance.

Next: Topic 4 -- Hallucination Detection (the deeper, semantic faithfulness check)
""")
